> **Versión corregida (v2.13, línea v1 de despliegue, ejecución verificada)** — generada a partir de `notebooks/13_despliegue_streamlit_ngrok_y_github_corregido.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización: se sustituyen la detección/montaje de Drive y el clonado de `main` por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`, rama `feature/entorno-capacidad-v2`).
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; `PERSISTENT_ROOT` se conserva como raíz de la línea v1 (= `PROJECT_ROOT`, o `PROJECT_ROOT/prueba_linea_v1` en modo prueba).
> - Secretos desde Colab (`userdata`) o variables de entorno; solo se exigen cuando se usan. Sin `NGROK_AUTHTOKEN` el dashboard se sirve localmente y se omite el túnel.
> - A1 instala solo dependencias faltantes; `pyngrok` solo si hay token de ngrok.
> - Nuevo `DASHBOARD_VERSION` (`v2` por defecto): la vista temporal sirve el dashboard principal V2 del repositorio o el v1 generado en el notebook 12.
> - Nueva celda A0: prueba de humo con `streamlit.testing` para ambos dashboards; A2 verifica además `/_stcore/health`.
> - B1 ya no sobrescribe `dashboard_streamlit/` (dashboard V2 versionado): el dashboard v1 se publica en `dashboard_streamlit_v1/` y los datos V2 en `dashboard_data/v2/`.
> - B1/B4 se omiten en modo prueba para no publicar resultados de prueba. B4 rechaza publicar en `main` y exige el token solo al publicar.
> - B3 compara contra `origin/<rama de trabajo>` en lugar de `origin/main`; B5 verifica también los archivos del dashboard V2 y v1; B6 documenta ambas rutas de despliegue.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña, pocos pasos, salida en `prueba_linea_v1/`).


# PIDA-RL Diabetes — 13. Despliegue de Streamlit con ngrok y GitHub

Este notebook permite visualizar el dashboard de dos maneras:

1. **Temporal:** Streamlit dentro de Google Colab expuesto mediante un túnel de ngrok.
2. **Persistente:** publicación en GitHub y despliegue posterior en Streamlit Community Cloud.

Los secretos ya configurados en Google Colab se leen sin imprimirlos:

- `NGROK_AUTHTOKEN`
- `GIT_USER_NAME`
- `GIT_USER_EMAIL`
- `TOKEN-pida-rl-datos-publicos`

> **Uso responsable:** publicar únicamente código y resultados agregados de simulación. No publicar datos identificables, expedientes, trazas individuales ni tokens.


## 1. Inicialización


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# --- Línea v1 de despliegue (entorno autocontenido) ---
# En modo prueba se escribe en una carpeta aparte para no sobrescribir resultados completos ni el dashboard publicado.
LINEA_V1_ROOT = PROJECT_ROOT / 'prueba_linea_v1' if MODO_PRUEBA else PROJECT_ROOT
PERSISTENT_ROOT = LINEA_V1_ROOT
COHORTE_CANONICA = PROJECT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
print('Raíz de la línea v1 (PERSISTENT_ROOT):', PERSISTENT_ROOT)

# --- Celda original (sin montaje de Drive ni clonado fijo) ---
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json
import socket
import time
import py_compile

DASHBOARD_DIR = PERSISTENT_ROOT / 'dashboard_streamlit'
DASHBOARD_DATA_DIR = PERSISTENT_ROOT / 'dashboard_data'
APP_PATH = DASHBOARD_DIR / 'app.py'
REQUIREMENTS_PATH = DASHBOARD_DIR / 'requirements.txt'

V2_APP_PATH = REPO_ROOT / 'dashboard_streamlit' / 'app.py'
V2_DASHBOARD_DIR = DASHBOARD_DATA_DIR / 'v2'
required_paths = [APP_PATH, REQUIREMENTS_PATH, DASHBOARD_DATA_DIR, V2_APP_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Faltan archivos o directorios producidos por el notebook 12:\n' + '\n'.join(missing_paths)
    )

os.chdir(REPO_ROOT)
print('Aplicación:', APP_PATH)
print('Datos:', DASHBOARD_DATA_DIR)
print('Repositorio:', REPO_ROOT)

# Dashboard a visualizar: 'v2' (principal, capacidad estricta) o 'v1' (línea autocontenida).
DASHBOARD_VERSION = os.environ.get('PIDA_DASHBOARD_VERSION', 'v2')
assert DASHBOARD_VERSION in {'v1', 'v2'}
print('Dashboard seleccionado:', DASHBOARD_VERSION)
print('Datos V2 disponibles:', (V2_DASHBOARD_DIR / 'manifest.csv').is_file())


## 2. Leer y validar secretos

Esta celda comprueba que Colab puede acceder a los secretos. No imprime valores privados. Si Colab solicita autorización, habilita **Notebook access** para este notebook.


In [ ]:
# Secretos: Colab (userdata) o variables de entorno. Ninguno se imprime.
# Solo se exigen cuando se usan: ngrok para la vista temporal y token para publicar.
SECRET_NAMES = {
    'ngrok': ('NGROK_AUTHTOKEN', 'NGROK_AUTHTOKEN'),
    'git_user_name': ('GIT_USER_NAME', 'GIT_USER_NAME'),
    'git_user_email': ('GIT_USER_EMAIL', 'GIT_USER_EMAIL'),
    'github_token': ('TOKEN-pida-rl-datos-publicos', 'GITHUB_TOKEN'),
}


def get_secret(key):
    colab_name, env_name = SECRET_NAMES[key]
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(colab_name)
            if value:
                return value
        except Exception as error:
            print(f'Aviso: no se pudo leer el secreto de Colab {colab_name}: {type(error).__name__}')
    return os.environ.get(env_name) or None


ngrok_authtoken = get_secret('ngrok')
git_user_name = get_secret('git_user_name')
git_user_email = get_secret('git_user_email')
github_token = get_secret('github_token')

for key, value in [('ngrok', ngrok_authtoken), ('identidad Git', git_user_name and git_user_email), ('token GitHub', github_token)]:
    print(f'{key:14s} | disponible: {bool(value)}')


## 3. Configurar identidad Git local

La identidad se configura solamente en el clon de `/content/pida-rl-datos-publicos`. No cambia la configuración global de Git y no revela los valores de los secretos.


In [ ]:
os.chdir(REPO_ROOT)

if git_user_name and git_user_email:
    subprocess.run(['git', 'config', '--local', 'user.name', git_user_name], check=True)
    subprocess.run(['git', 'config', '--local', 'user.email', git_user_email], check=True)
    print('Identidad Git local configurada correctamente.')
else:
    print('Identidad Git no configurada: solo es necesaria para la Versión B (publicación).')


# Versión A — ngrok

La URL de ngrok será temporal. El dashboard dejará de estar disponible al cerrar el túnel, detener Streamlit o reiniciar el entorno de Colab.


## A1. Instalar dependencias


In [ ]:
# Instala solo lo que falte (Colab ya trae pandas; Streamlit y Plotly suelen faltar).
faltantes = [pip for pip, mod in [('streamlit', 'streamlit'), ('plotly', 'plotly'), ('pandas', 'pandas')]
             if importlib.util.find_spec(mod) is None]
if ngrok_authtoken and importlib.util.find_spec('pyngrok') is None:
    faltantes.append('pyngrok')
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltantes], check=True)
print('Dependencias de Streamlit y Plotly disponibles. pyngrok:', importlib.util.find_spec('pyngrok') is not None)


## A0. Prueba de humo del dashboard

Renderiza las aplicaciones v1 y V2 con `streamlit.testing` (sin servidor ni túnel) y falla si alguna muestra excepciones o mensajes de error.


In [ ]:
# Prueba de humo: renderiza ambas aplicaciones sin servidor ni túnel (streamlit.testing).
from streamlit.testing.v1 import AppTest

def probar_app(app_path, data_dir=None):
    anterior = os.environ.get('PIDA_DASHBOARD_DATA_DIR')
    if data_dir is not None:
        os.environ['PIDA_DASHBOARD_DATA_DIR'] = str(data_dir)
    try:
        at = AppTest.from_file(str(app_path), default_timeout=120).run()
    finally:
        if data_dir is not None:
            if anterior is None:
                os.environ.pop('PIDA_DASHBOARD_DATA_DIR', None)
            else:
                os.environ['PIDA_DASHBOARD_DATA_DIR'] = anterior
    errores = [e.value for e in at.exception] + [e.value for e in at.error]
    return {'app': str(Path(app_path).relative_to(Path(app_path).parents[1])), 'excepciones': len(at.exception),
            'errores': len(at.error), 'encabezados': len(at.header) + len(at.title), 'detalle': '; '.join(map(str, errores))[:300]}

import pandas as pd
resultados_humo = [probar_app(APP_PATH)]
if (V2_DASHBOARD_DIR / 'manifest.csv').is_file():
    resultados_humo.append(probar_app(V2_APP_PATH, V2_DASHBOARD_DIR))
df_humo = pd.DataFrame(resultados_humo)
display(df_humo)
assert (df_humo['excepciones'] == 0).all() and (df_humo['errores'] == 0).all(), 'El dashboard muestra errores.'
print('Prueba de humo aprobada.')


## A2. Validar e iniciar dashboard

Primero se valida que `app.py` compile. Después se inicia Streamlit y se abre un túnel HTTP hacia el puerto 8501.


In [ ]:
APP_SELECCIONADA = V2_APP_PATH if DASHBOARD_VERSION == 'v2' else APP_PATH
py_compile.compile(str(APP_SELECCIONADA), doraise=True)
PORT = 8501


def port_is_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)

env_app = dict(os.environ)
if DASHBOARD_VERSION == 'v2':
    env_app['PIDA_DASHBOARD_DATA_DIR'] = str(V2_DASHBOARD_DIR)

streamlit_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(APP_SELECCIONADA), '--server.port', str(PORT),
     '--server.address', '0.0.0.0', '--server.headless', 'true', '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, env=env_app, cwd=str(REPO_ROOT),
)

for _ in range(60):
    if port_is_open('127.0.0.1', PORT):
        break
    time.sleep(1)
else:
    raise RuntimeError('Streamlit no abrió el puerto 8501 en el tiempo esperado.')

import urllib.request
for _ in range(30):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/_stcore/health', timeout=5) as response:
            salud = response.read().decode().strip()
        break
    except Exception:
        time.sleep(1)
print('Salud de Streamlit:', salud)
assert salud == 'ok'
print(f'Dashboard {DASHBOARD_VERSION} activo: http://127.0.0.1:{PORT}')

if ngrok_authtoken:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_authtoken)
    if 'public_tunnel' in globals():
        try:
            ngrok.disconnect(public_tunnel.public_url)
        except Exception:
            pass
    public_tunnel = ngrok.connect(addr=PORT, proto='http')
    print('Abre esta URL temporal de ngrok:', public_tunnel.public_url)
else:
    print('Sin NGROK_AUTHTOKEN: se omite el túnel; el dashboard queda disponible solo de forma local.')


## A3. Cerrar la vista temporal


In [ ]:
if 'public_tunnel' in globals() and ngrok_authtoken:
    from pyngrok import ngrok
    try:
        ngrok.disconnect(public_tunnel.public_url)
        print('Túnel de ngrok cerrado.')
    except Exception as error:
        print('No fue posible cerrar el túnel o ya estaba cerrado:', error)

if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
    print('Proceso de Streamlit detenido.')


# Versión B — GitHub y Streamlit Community Cloud

Esta sección copia solo el código del dashboard y los resultados agregados a la copia local del repositorio. La publicación requiere confirmación explícita mediante el valor `PUBLICAR`.


## B1. Copiar archivos permitidos al repositorio


In [ ]:
REPO_APP_V1_DIR = REPO_ROOT / 'dashboard_streamlit_v1'
REPO_DATA_DIR = REPO_ROOT / 'dashboard_data'
PUBLICACION_PERMITIDA = not MODO_PRUEBA

allowed_data_files = [
    'tabla_decision_final_v1.csv',
    'cobertura_por_riesgo_dashboard_v1.csv',
    'alto_riesgo_por_semilla_dashboard_v1.csv',
    'conclusiones_dashboard_v1.csv',
    'configuracion_dashboard_v1.json',
]
allowed_v1_app_files = ['app.py', 'requirements.txt', 'README.md']

if not PUBLICACION_PERMITIDA:
    print('Modo prueba: no se copian resultados de prueba al repositorio (B1 y B4 se omiten).')
else:
    # El dashboard V2 (dashboard_streamlit/) es código versionado y no se sobrescribe.
    REPO_APP_V1_DIR.mkdir(parents=True, exist_ok=True)
    for name in allowed_v1_app_files:
        shutil.copy2(DASHBOARD_DIR / name, REPO_APP_V1_DIR / name)
    REPO_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in allowed_data_files:
        shutil.copy2(DASHBOARD_DATA_DIR / name, REPO_DATA_DIR / name)
    if (V2_DASHBOARD_DIR / 'manifest.csv').is_file():
        shutil.copytree(V2_DASHBOARD_DIR, REPO_DATA_DIR / 'v2', dirs_exist_ok=True)

    required_repo_files = (
        [REPO_APP_V1_DIR / n for n in allowed_v1_app_files]
        + [REPO_DATA_DIR / n for n in allowed_data_files]
        + [REPO_DATA_DIR / 'v2' / 'manifest.csv', REPO_ROOT / 'dashboard_streamlit' / 'app.py']
    )
    missing_repo_files = [str(p.relative_to(REPO_ROOT)) for p in required_repo_files if not p.is_file()]
    if missing_repo_files:
        raise FileNotFoundError('Faltan archivos tras la copia al repositorio:\n' + '\n'.join(missing_repo_files))
    print('Archivos permitidos preparados para GitHub:')
    for p in required_repo_files:
        print('-', p.relative_to(REPO_ROOT), '| bytes:', p.stat().st_size)


## B2. Revisar cambios

Revisa esta salida antes de publicar. Deben aparecer solamente `dashboard_streamlit/` y `dashboard_data/`, con archivos agregados y sin información sensible.


In [ ]:
os.chdir(REPO_ROOT)

status = subprocess.run(
    ['git', 'status', '--branch', '--short'],
    text=True,
    capture_output=True,
    check=True,
)

print(status.stdout if status.stdout.strip() else 'No hay cambios pendientes.')

tracked_dashboard_files = subprocess.run(
    ['git', 'ls-files', 'dashboard_streamlit', 'dashboard_streamlit_v1', 'dashboard_data'],
    text=True,
    capture_output=True,
    check=True,
)

if tracked_dashboard_files.stdout.strip():
    print('\nArchivos ya versionados del dashboard:')
    print(tracked_dashboard_files.stdout.strip())


## B3. Corregir autor del último commit local, si aplica

Esta celda es opcional. Solo corrige la autoría si el último commit local fue creado con `Your Name <you@example.com>` y aún no fue enviado a GitHub. No cambia los archivos del dashboard.


In [ ]:
os.chdir(REPO_ROOT)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=False, capture_output=True)

last_commit = subprocess.run(
    ['git', 'log', '-1', '--format=%H|%an|%ae|%s'],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

commit_hash, author_name, author_email, subject = last_commit.split('|', 3)

ahead_count = subprocess.run(
    ['git', 'rev-list', '--count', f'origin/{BRANCH}..HEAD'],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

should_amend = (
    ahead_count != '0'
    and author_name == 'Your Name'
    and author_email == 'you@example.com'
)

if should_amend:
    subprocess.run(
        ['git', 'commit', '--amend', '--reset-author', '--no-edit'],
        check=True,
)
    print('Autor del último commit local corregido con los secretos de Colab.')
else:
    print('No se requiere corregir la autoría del último commit.')


## B4. Confirmar y publicar en GitHub

Escribe exactamente `PUBLICAR` para autorizar un `git add`, `git commit` si hay cambios y `git push` a la rama actual. La celda no imprime el token.


In [ ]:
CONFIRMACION_PUBLICACION = 'NO'

PUBLICADO = False
if not PUBLICACION_PERMITIDA:
    print('Modo prueba: no se publica en GitHub.')
elif CONFIRMACION_PUBLICACION != 'PUBLICAR':
    print('No se realizó commit ni push. Cambia CONFIRMACION_PUBLICACION a PUBLICAR solo después de revisar B2.')
else:
    os.chdir(REPO_ROOT)

    subprocess.run(
        ['git', 'add', 'dashboard_streamlit_v1', 'dashboard_data'],
        check=True,
)

    staged = subprocess.run(
        ['git', 'diff', '--cached', '--name-only'],
        text=True,
        capture_output=True,
        check=True,
).stdout.strip()

    if staged:
        print('Archivos preparados para commit:')
        print(staged)

        commit = subprocess.run(
            ['git', 'commit', '-m', 'Agregar dashboard Streamlit y resultados agregados de simulación'],
            text=True,
            capture_output=True,
)

        if commit.returncode != 0:
            raise RuntimeError(
                'El commit falló. Mensaje de Git:\n'
                + (commit.stdout or '')
                + (commit.stderr or '')
            )

        print(commit.stdout.strip())
    else:
        print('No hay cambios nuevos para crear un commit.')

    branch = subprocess.run(
        ['git', 'branch', '--show-current'],
        text=True,
        capture_output=True,
        check=True,
).stdout.strip()

    if not branch:
        raise RuntimeError(f'No se detectó una rama activa. Ejecuta git switch {BRANCH} antes de publicar.')
    if branch == 'main':
        raise RuntimeError('Política del proyecto: no se publica directamente en main; usa la rama de trabajo y un pull request.')
    if not github_token:
        raise RuntimeError('Falta el token de GitHub (secreto TOKEN-pida-rl-datos-publicos o variable GITHUB_TOKEN).')

    token_remote = (
        'https://x-access-token:'
        + github_token
        + '@github.com/JssSalas/pida-rl-datos-publicos.git'
)

    push = subprocess.run(
        ['git', 'push', token_remote, branch],
        text=True,
        capture_output=True,
)

    if push.returncode != 0:
        raise RuntimeError(
            'El push falló. Verifica que el token fine-grained tenga acceso al repositorio '
            'y permiso Contents: Read and write. Mensaje de Git:\n'
            + (push.stdout or '')
            + (push.stderr or '')
        )

    print(push.stdout.strip())
    print('Publicación completada en GitHub, rama:', branch)
    PUBLICADO = True


## B5. Verificación posterior al push


In [ ]:
os.chdir(REPO_ROOT)

status = subprocess.run(
    ['git', 'status', '--branch', '--short'],
    text=True,
    capture_output=True,
    check=True,
)
print(status.stdout if status.stdout.strip() else 'Estado limpio.')

files_in_head = subprocess.run(
    ['git', 'ls-tree', '-r', '--name-only', 'HEAD'],
    text=True,
    capture_output=True,
    check=True,
).stdout.splitlines()

expected_github_paths = [
    'dashboard_streamlit/app.py',
    'dashboard_streamlit/data_loader.py',
    'dashboard_streamlit/requirements.txt',
    'dashboard_streamlit_v1/app.py',
    'dashboard_streamlit_v1/requirements.txt',
    'dashboard_data/v2/manifest.csv',
    'dashboard_data/tabla_decision_final_v1.csv',
    'dashboard_data/cobertura_por_riesgo_dashboard_v1.csv',
    'dashboard_data/alto_riesgo_por_semilla_dashboard_v1.csv',
    'dashboard_data/conclusiones_dashboard_v1.csv',
    'dashboard_data/configuracion_dashboard_v1.json',
]

missing_in_head = [path for path in expected_github_paths if path not in files_in_head]
print('Archivos del despliegue presentes en HEAD:', len(expected_github_paths) - len(missing_in_head), 'de', len(expected_github_paths))
if missing_in_head and globals().get('PUBLICADO'):
    raise FileNotFoundError('Faltan archivos en el último commit local:\n' + '\n'.join(missing_in_head))

print('Pendientes en HEAD:' if missing_in_head else 'Verificación local aprobada. Archivos requeridos presentes en HEAD.', missing_in_head or '')


## B6. Desplegar en Streamlit Community Cloud

Después de ejecutar B4 exitosamente (o con el contenido ya publicado en la rama):

1. Abre [Streamlit Community Cloud](https://share.streamlit.io/) e inicia sesión con GitHub.
2. Selecciona **Create app → Deploy a public app from GitHub**.
3. Repositorio: `JssSalas/pida-rl-datos-publicos`.
4. Rama: `feature/entorno-capacidad-v2` (o `main` cuando el pull request se haya integrado).
5. **Main file path**:

| Dashboard | Ruta | Datos |
|---|---|---|
| Principal V2 (capacidad estricta) | `dashboard_streamlit/app.py` | `dashboard_data/v2/` |
| Línea v1 autocontenida | `dashboard_streamlit_v1/app.py` | `dashboard_data/*_v1.*` |

6. Selecciona **Deploy**. Cada aplicación usa el `requirements.txt` de su propia carpeta.


## Resultado esperado

- A2 imprime una URL temporal de ngrok para visualizar Streamlit desde Colab.
- B4 publica las carpetas `dashboard_streamlit/` y `dashboard_data/` después de confirmar `PUBLICAR`.
- B6 despliega la aplicación persistente usando `dashboard_streamlit/app.py`.
